# YouTube spot check v2: numbers written as words

Version 1 (`youtube_spot_check.ipynb`) scored the four systems with `clean_urdu` v1, the normaliser used by every experiment.
`clean_urdu` v1 does not touch numbers. On this set that is unfair to some systems:

| Text | Words containing digits |
|---|---|
| Gold transcripts | 2 |
| Indic Transcribe Core | 0 |
| Whisper-large-v3 | 12 |
| Omnilingual LLM 1B v2 | 40 |
| Awaaz se Alfaaz system (`model_transcript`) | 64 |

The gold writes numbers as words (`80%` is اسی فیصد, `5,222` is پانچ ہزار دو سو بائیس), so a system that writes the
right number in digits is counted wrong for every word of it. This notebook adds one step, **numbers as words**, before `clean_urdu` v1,
applied in the same way to the gold and to all four systems, and scores everything twice:

| Normaliser | Use |
|---|---|
| `clean_urdu v1` | Same as experiments 01 to 06. Use this when comparing with other experiments. It must reproduce the version 1 scores exactly (checked below) |
| `clean_urdu v1 + numbers as words` | Fairer comparison of the four systems with each other on this set |

Nothing is transcribed again. The notebook reads the transcripts that version 1 saved, so it needs no GPU, audio or internet.
The dataset card does not say how its authors normalised text, so their own published WER is still not directly comparable.

Outputs (version 1 files are not changed):

| File | Contents |
|---|---|
| `results/transcripts/all_systems_by_clip_v2.csv` | All systems per clip, under both normalisers, with per-clip WER and CER |
| `results/scores/youtube_spot_check_v2_scores.csv` | Same columns as version 1, plus `normaliser` |
| `results/scores/youtube_spot_check_v2_paired_bootstrap.csv` | Same columns as version 1, plus `normaliser` |
| `results/scores/youtube_spot_check_v2_per_clip.csv` | Per-clip WER and CER of every system under both normalisers |
| `results/scores/youtube_spot_check_v2_digit_words.csv` | Words containing digits before and after the number step |
| `results/run_info/youtube_spot_check_v2.json` | Normalisers, number rules, package versions |

## 1. Setup

In [ ]:
import json, re, string, unicodedata, platform, sys
from itertools import combinations
from pathlib import Path
import numpy as np, pandas as pd, jiwer

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PROMPTS.md").exists())
EXP = REPO / "experiments" / "07_youtube_spot_check"
DATA, RESULTS = EXP / "data", EXP / "results"

SYSTEMS = {
    "whisper_large_v3": "Whisper-large-v3",
    "omnilingual_llm_1b_v2": "Omnilingual LLM 1B v2",
    "indic_transcribe_core": "Indic Transcribe Core",
    "awaaz_model_transcript": "Awaaz se Alfaaz system (dataset's model_transcript)",
}
RECOGNISERS = list(SYSTEMS)[:3]
DATASET = "Awaaz se Alfaaz YouTube"

# clean_urdu v1, copied unchanged from experiment 01
_DIAC_RE = re.compile("[ؐ-ؚـً-ٰۖ-ۜ۟-۪ۨ-ۭ]")
_PUNCT = set(string.punctuation) | set("۔،؟؛٪।॥“”"
                                       "‘’—–…«»‖‐‒·")
_IZAFAT = {"اے", "ای", "ئے", "ء", "ے", "او"}
_UMAP = {"ي": "ی", "ك": "ک", "ة": "ہ", "ۃ": "ہ",
         "ى": "ی", "إ": "ا", "أ": "ا", "آ": "ا",
         "ٲ": "ا", "ٳ": "ا", "ؤ": "و", "ئ": "ی"}

def clean_urdu(text):
    t = unicodedata.normalize("NFC", str(text))
    t = _DIAC_RE.sub("", t)
    t = "".join(_UMAP.get(ch, ch) for ch in t)
    t = "".join(" " if ch in _PUNCT else ch for ch in t)
    return " ".join(w for w in t.split() if w not in _IZAFAT).strip()

## 2. Numbers as words

The rules follow how the gold transcripts write numbers:

| Written | Becomes | As in the gold |
|---|---|---|
| `24`, `81` | چوبیس, اکیاسی | چوبیس رنز, اکیاسی فیصد |
| `104`, `202` | ایک سو چار, دو سو دو | ایک سو چار رنز |
| `5,222`, `10،000` (comma or Urdu comma between groups of 3) | پانچ ہزار دو سو بائیس, دس ہزار | پانچ ہزار دو سو بائیس |
| `2026` | دو ہزار چھبیس | دو ہزار چھبیس |
| `4.5` | چار اعشاریہ پانچ (digits after the point read one by one) | چار اعشاریہ پانچ |
| `80%`, `80٪` | اسی فیصد | اسی فیصد |
| `19ویں` | انیسویں (ordinal ending stays joined) | انیسویں |

Larger numbers use لاکھ, کروڑ and ارب. Urdu digits (۰ to ۹) and Arabic digits (٠ to ٩) are read the same as 0 to 9.
A number starting with 0 (such as `007`), or longer than 12 digits, is read digit by digit.
Any other letters stuck to a number (`2024ء`, `10بجے`) are split off with a space.

Known limits: numbers spoken in English stay different (the gold writes flight `709` as سیون زیرو نائن), and different spellings of
the same number word (چھ, چھے) are not merged. Both affect every system the same way.

In [ ]:
_UNITS = ("صفر ایک دو تین چار پانچ چھ سات آٹھ نو دس گیارہ بارہ تیرہ چودہ پندرہ سولہ سترہ اٹھارہ انیس بیس "
          "اکیس بائیس تئیس چوبیس پچیس چھبیس ستائیس اٹھائیس انتیس تیس اکتیس بتیس تینتیس چونتیس پینتیس چھتیس سینتیس اڑتیس انتالیس چالیس "
          "اکتالیس بیالیس تینتالیس چوالیس پینتالیس چھیالیس سینتالیس اڑتالیس انچاس پچاس اکیاون باون ترپن چون پچپن چھپن ستاون اٹھاون انسٹھ ساٹھ "
          "اکسٹھ باسٹھ تریسٹھ چونسٹھ پینسٹھ چھیاسٹھ سڑسٹھ اڑسٹھ انہتر ستر اکہتر بہتر تہتر چوہتر پچھتر چھہتر ستتر اٹھہتر اناسی اسی "
          "اکیاسی بیاسی تراسی چوراسی پچاسی چھیاسی ستاسی اٹھاسی نواسی نوے اکیانوے بانوے ترانوے چورانوے پچانوے چھیانوے ستانوے اٹھانوے ننانوے").split()
assert len(_UNITS) == 100
_SCALES = [(10**9, "ارب"), (10**7, "کروڑ"), (10**5, "لاکھ"), (1000, "ہزار"), (100, "سو")]
_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
_ORDINAL = ("ویں", "واں", "ویاں")
_NUMBER_RE = re.compile(r"(\d{1,3}(?:[,،]\d{3})+|\d+)(?:\.(\d+))?(\s*[%٪])?(ویاں|ویں|واں)?")


def int_to_urdu(n):
    if n < 100:
        return _UNITS[n]
    parts = []
    for value, name in _SCALES:
        if n >= value:
            q, n = divmod(n, value)
            parts.append(f"{int_to_urdu(q)} {name}")
    if n:
        parts.append(_UNITS[n])
    return " ".join(parts)


def digits_one_by_one(s):
    return " ".join(_UNITS[int(d)] for d in s)


def _number(m):
    whole, frac, percent, ordinal = m.group(1), m.group(2), m.group(3), m.group(4)
    whole = re.sub("[,،]", "", whole)
    words = digits_one_by_one(whole) if (len(whole) > 1 and whole[0] == "0") or len(whole) > 12 else int_to_urdu(int(whole))
    if frac:
        words += " اعشاریہ " + digits_one_by_one(frac)
    if ordinal:
        words += ordinal              # 19ویں -> انیسویں
    if percent:
        words += " فیصد"
    return f" {words} "


def numbers_as_words(text):
    return _NUMBER_RE.sub(_number, unicodedata.normalize("NFC", str(text)).translate(_DIGITS))


def clean_urdu_numbers(text):
    return clean_urdu(numbers_as_words(text))


NORMALISERS = {"clean_urdu v1": clean_urdu, "clean_urdu v1 + numbers as words": clean_urdu_numbers}

# Checks against how the gold writes these numbers
for raw, gold in [("80% سے زائد", "اسی فیصد سے زائد"), ("5,222 سے زیادہ", "پانچ ہزار دو سو بائیس سے زیادہ"),
                  ("10،000 کم", "دس ہزار کم"), ("رمضان المبارک 2026 کی", "رمضان المبارک دو ہزار چھبیس کی"),
                  ("81,365 تھی", "اکیاسی ہزار تین سو پینسٹھ تھی"), ("4.5 ملین", "چار اعشاریہ پانچ ملین"),
                  ("104 رنز", "ایک سو چار رنز"), ("19ویں اوور", "انیسویں اوور"), ("100 پاؤنڈ", "ایک سو پاؤنڈ"),
                  ("2024ء میں", "دو ہزار چوبیس میں"), ("۲۱ نومبر", "اکیس نومبر"), ("150000", "ایک لاکھ پچاس ہزار")]:
    got = clean_urdu_numbers(raw)
    assert got == clean_urdu(gold), (raw, got, clean_urdu(gold))
print("number rules pass all checks")

## 3. Transcripts from version 1

Reads the gold, the dataset's `model_transcript` and the three recognisers' transcripts saved by version 1.

In [ ]:
clips = pd.read_csv(DATA / "samples_youtube_eval.csv", keep_default_na=False)
clips["downloaded"] = clips.downloaded.astype(str) == "True"
clips["duration_s"] = pd.to_numeric(clips.duration_s, errors="coerce")
by_clip = clips[["clip_id", "video_name", "downloaded", "duration_s", "segments", "reference"]].copy()

for stem in RECOGNISERS:
    t = pd.read_csv(RESULTS / "transcripts" / f"{stem}_transcripts.csv", keep_default_na=False).set_index("clip_id")
    missing = set(clips.clip_id[clips.downloaded]) - set(t.index)
    assert not missing, f"{SYSTEMS[stem]} has no transcript for {len(missing)} clips; run version 1 first"
    t = t.reindex(by_clip.clip_id)
    by_clip[stem] = t.transcript.fillna("").values
    by_clip[f"{stem}_time_s"] = pd.to_numeric(t.time_s, errors="coerce").values
    by_clip[f"{stem}_status"] = t.status.fillna("not downloaded").values
by_clip["awaaz_model_transcript"] = clips.model_transcript.values
by_clip["awaaz_model_transcript_time_s"] = np.nan
by_clip["awaaz_model_transcript_status"] = "ok"
print(f"{by_clip.downloaded.sum()} of {len(by_clip)} clips have audio and recogniser transcripts")

## 4. Scores under both normalisers

Same scoring as version 1 and experiment 01: corpus WER and CER, and S / D / I as % of reference words.
A failed or empty transcript counts every reference word as a deletion.

In [ ]:
def edit_counts(ref, hyp):
    # Returns (S, D, I, reference words, character edits, reference characters).
    if not hyp:
        return 0, len(ref.split()), 0, len(ref.split()), len(ref), len(ref)
    w = jiwer.process_words(ref, hyp)
    c = jiwer.process_characters(ref, hyp)
    return (w.substitutions, w.deletions, w.insertions, w.hits + w.substitutions + w.deletions,
            c.substitutions + c.deletions + c.insertions, c.hits + c.substitutions + c.deletions)


TAG = {"clean_urdu v1": "v1", "clean_urdu v1 + numbers as words": "num"}
COUNTS, SCORABLE = {}, {}
for norm, fn in NORMALISERS.items():
    tag = TAG[norm]
    by_clip[f"reference_{tag}"] = by_clip.reference.map(fn)
    scorable = by_clip[f"reference_{tag}"].str.len() > 0
    SCORABLE[norm] = scorable
    for stem in SYSTEMS:
        by_clip[f"{stem}_{tag}"] = by_clip[stem].map(fn)
        ok = scorable & (by_clip.downloaded if stem in RECOGNISERS else True)
        counts = np.array([edit_counts(r, h) if k else (0, 0, 0, 0, 0, 0)
                           for r, h, k in zip(by_clip[f"reference_{tag}"], by_clip[f"{stem}_{tag}"], ok)])
        COUNTS[norm, stem] = counts
        with np.errstate(invalid="ignore", divide="ignore"):
            by_clip[f"{stem}_WER_%_{tag}"] = np.where(ok, np.round(100 * counts[:, :3].sum(1) / counts[:, 3], 2), np.nan)
            by_clip[f"{stem}_CER_%_{tag}"] = np.where(ok, np.round(100 * counts[:, 4] / counts[:, 5], 2), np.nan)
by_clip.to_csv(RESULTS / "transcripts" / "all_systems_by_clip_v2.csv", index=False, encoding="utf-8-sig")

SUBSETS = [("downloaded clips", by_clip.downloaded, list(SYSTEMS))]
if not by_clip.downloaded.all():
    SUBSETS.append(("all 30 clips", pd.Series(True, index=by_clip.index), ["awaaz_model_transcript"]))

rows = []
for norm in NORMALISERS:
    scorable = SCORABLE[norm]
    reference_words = by_clip[f"reference_{TAG[norm]}"].str.split().str.len()
    filters = [("all clips", scorable), ("references with 3+ words", scorable & (reference_words >= 3))]
    for subset, in_subset, systems in SUBSETS:
        for filter_name, keep in filters:
            m = (in_subset & keep).values
            for stem in systems:
                c = COUNTS[norm, stem][m].sum(0)
                times = by_clip.loc[m, f"{stem}_time_s"]
                rows.append({"dataset": DATASET, "subset": subset, "clips_scored": filter_name, "normaliser": norm,
                             "clips": int(m.sum()), "reference_words": int(c[3]), "recogniser": SYSTEMS[stem],
                             "WER_%": round(100 * c[:3].sum() / c[3], 2), "CER_%": round(100 * c[4] / c[5], 2),
                             "substitutions_%": round(100 * c[0] / c[3], 2), "deletions_%": round(100 * c[1] / c[3], 2),
                             "insertions_%": round(100 * c[2] / c[3], 2),
                             "failed_or_empty": int((by_clip.loc[m, f"{stem}_status"] != "ok").sum()),
                             "total_time_min": round(times.sum() / 60, 1) if times.notna().any() else np.nan,
                             "seconds_per_clip": round(times.mean(), 2) if times.notna().any() else np.nan})
scores = pd.DataFrame(rows)

# clean_urdu v1 must give exactly the version 1 scores
v1_saved = pd.read_csv(RESULTS / "scores" / "youtube_spot_check_scores.csv")
key = ["subset", "clips_scored", "recogniser"]
check = scores[scores.normaliser == "clean_urdu v1"].merge(v1_saved, on=key, suffixes=("", "_saved"))
assert len(check) == len(v1_saved) and np.allclose(check["WER_%"], check["WER_%_saved"]) \
    and np.allclose(check["CER_%"], check["CER_%_saved"]), "clean_urdu v1 scores differ from version 1"
print("clean_urdu v1 reproduces the version 1 scores exactly")

scores.to_csv(RESULTS / "scores" / "youtube_spot_check_v2_scores.csv", index=False, encoding="utf-8-sig")
show = ["recogniser", "clips", "reference_words", "WER_%", "CER_%", "substitutions_%", "deletions_%", "insertions_%"]
for norm in NORMALISERS:
    t = scores[(scores.normaliser == norm) & (scores.subset == "downloaded clips") & (scores.clips_scored == "all clips")]
    print(f"\n{DATASET} | downloaded clips | {norm}")
    print(t[show].sort_values("WER_%").to_string(index=False))

### Change caused by the number step

In [ ]:
base = scores[(scores.subset == "downloaded clips") & (scores.clips_scored == "all clips")]
change = base.pivot(index="recogniser", columns="normaliser", values="WER_%")
change["WER change"] = (change["clean_urdu v1 + numbers as words"] - change["clean_urdu v1"]).round(2)
change["rank v1"] = change["clean_urdu v1"].rank().astype(int)
change["rank with numbers"] = change["clean_urdu v1 + numbers as words"].rank().astype(int)
print(change.sort_values("clean_urdu v1 + numbers as words").to_string())

has_digit = lambda t: sum(any(ch.isdigit() for ch in w) for w in str(t).split())
d = by_clip[by_clip.downloaded]
digits = pd.DataFrame([{"text": "gold reference",
                        "words_with_digits_v1": int(d.reference_v1.map(has_digit).sum()),
                        "words_with_digits_numbers_as_words": int(d.reference_num.map(has_digit).sum())}] +
                      [{"text": name,
                        "words_with_digits_v1": int(d[f"{s}_v1"].map(has_digit).sum()),
                        "words_with_digits_numbers_as_words": int(d[f"{s}_num"].map(has_digit).sum())}
                       for s, name in SYSTEMS.items()])
digits.to_csv(RESULTS / "scores" / "youtube_spot_check_v2_digit_words.csv", index=False, encoding="utf-8-sig")
print()
print(digits.to_string(index=False))

## 5. Paired bootstrap

As in version 1: for each pair of systems, on the same clips, 1,000 resamples of the clips with replacement (seed 42);
the middle 95% of WER(A) − WER(B) is the 95% interval. Done under both normalisers.

In [ ]:
def paired_bootstrap(errors_a, errors_b, words, n=1000, seed=42):
    idx = np.random.default_rng(seed).integers(0, len(words), size=(n, len(words)))
    diff = 100 * (errors_a[idx].sum(1) - errors_b[idx].sum(1)) / words[idx].sum(1)
    return np.percentile(diff, [2.5, 97.5])

rows = []
for norm in NORMALISERS:
    scorable = SCORABLE[norm]
    reference_words = by_clip[f"reference_{TAG[norm]}"].str.split().str.len()
    filters = [("all clips", scorable), ("references with 3+ words", scorable & (reference_words >= 3))]
    for subset, in_subset, systems in SUBSETS:
        for filter_name, keep in filters:
            m = (in_subset & keep).values
            for a, b in combinations(systems, 2):
                ea, eb = COUNTS[norm, a][m, :3].sum(1), COUNTS[norm, b][m, :3].sum(1)
                words = COUNTS[norm, a][m, 3]
                lo, hi = paired_bootstrap(ea, eb, words)
                wa, wb = 100 * ea.sum() / words.sum(), 100 * eb.sum() / words.sum()
                verdict = (f"{SYSTEMS[a]} lower" if hi < 0 else f"{SYSTEMS[b]} lower" if lo > 0
                           else "no reliable difference")
                rows.append({"dataset": DATASET, "subset": subset, "clips_scored": filter_name, "normaliser": norm,
                             "clips": int(m.sum()), "recogniser_A": SYSTEMS[a], "recogniser_B": SYSTEMS[b],
                             "WER_A_%": round(wa, 2), "WER_B_%": round(wb, 2), "difference_A_minus_B": round(wa - wb, 2),
                             "interval_low": round(lo, 2), "interval_high": round(hi, 2), "result": verdict})
boot = pd.DataFrame(rows)
boot.to_csv(RESULTS / "scores" / "youtube_spot_check_v2_paired_bootstrap.csv", index=False, encoding="utf-8-sig")

show = ["recogniser_A", "recogniser_B", "WER_A_%", "WER_B_%", "difference_A_minus_B", "interval_low", "interval_high", "result"]
for norm in NORMALISERS:
    print(f"\n{norm}")
    print(boot[(boot.normaliser == norm) & (boot.subset == "downloaded clips") & (boot.clips_scored == "all clips")][show].to_string(index=False))

## 6. Per clip

In [ ]:
d = by_clip[by_clip.downloaded]
per_clip = d[["clip_id", "video_name", "duration_s", "segments"]].copy()
for s, name in SYSTEMS.items():
    short = name.split(" (")[0]
    for tag, label in [("v1", "v1"), ("num", "numbers as words")]:
        per_clip[f"{short} WER % ({label})"] = d[f"{s}_WER_%_{tag}"]
        per_clip[f"{short} CER % ({label})"] = d[f"{s}_CER_%_{tag}"]
per_clip.to_csv(RESULTS / "scores" / "youtube_spot_check_v2_per_clip.csv", index=False, encoding="utf-8-sig")

best = pd.DataFrame({tag: d[[f"{s}_WER_%_{tag}" for s in SYSTEMS]].idxmin(axis=1)
                          .str.replace(f"_WER_%_{tag}", "", regex=False).map(SYSTEMS).value_counts()
                     for tag in ["v1", "num"]}).fillna(0).astype(int)
best.columns = ["best on clips (v1)", "best on clips (numbers as words)"]
print(best.to_string())

moved = d.assign(change=d["awaaz_model_transcript_WER_%_num"] - d["awaaz_model_transcript_WER_%_v1"])
print("\nClips where the number step changes the Awaaz system's WER most:")
print(moved.sort_values("change")[["video_name", "awaaz_model_transcript_WER_%_v1",
                                   "awaaz_model_transcript_WER_%_num", "change"]].head(8).round(2).to_string(index=False))

## 7. Run information

In [ ]:
info = {
    "based_on": "youtube_spot_check.ipynb (version 1); transcripts read from its results, nothing transcribed again",
    "normalisers": {
        "clean_urdu v1": "same as experiments 01 to 06",
        "clean_urdu v1 + numbers as words": "numbers_as_words() before clean_urdu v1, on gold and all systems",
    },
    "number_rules": {
        "integers": "Urdu words 0 to 99; سو, ہزار, لاکھ, کروڑ, ارب; 100 = ایک سو",
        "group_separators": "comma or Urdu comma between groups of 3 digits are removed",
        "decimals": "اعشاریہ, then the digits after the point one by one",
        "percent": "% or ٪ after a number becomes فیصد",
        "ordinal_endings": list(_ORDINAL),
        "digit_by_digit": "numbers starting with 0, or longer than 12 digits",
        "other_digits": "Urdu ۰-۹ and Arabic ٠-٩ read as 0-9",
    },
    "bootstrap": {"resamples": 1000, "seed": 42},
    "python": platform.python_version(),
    "packages": {"pandas": pd.__version__, "numpy": np.__version__,
                 "jiwer": getattr(jiwer, "__version__", "unknown")},
}
(RESULTS / "run_info" / "youtube_spot_check_v2.json").write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
for f in sorted([*RESULTS.rglob("youtube_spot_check_v2*"), RESULTS / "transcripts" / "all_systems_by_clip_v2.csv"]):
    print("saved", f.relative_to(REPO))